In [2]:
import os
import sys

import pandas as pd
import numpy as np
import anndata as ad

import tacco as tc

# The notebook expects to be executed either in the workflow directory or in the repository root folder...
sys.path.insert(1, os.path.abspath('workflow' if os.path.exists('workflow/common_code.py') else '..')) 
import common_code

In [3]:
import numpy as np
import scipy.sparse as sp

def make_float32(adata):
    # If it's a view, materialize it
    if adata.is_view:
        adata = adata.copy()

    # If the AnnData was opened in backed mode, reload into memory
    if getattr(adata, "isbacked", False):
        import anndata as ad
        adata = ad.read_h5ad(adata.filename, backed=None)

    # Convert X
    if sp.issparse(adata.X):
        adata.X = adata.X.asfptype()           # to float
        adata.X = adata.X.astype(np.float32)   # to float32
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)

    return adata

In [ ]:
import os.path as osp 
import scanpy as sc
rna_path = '/maiziezhou_lab2/yuling/MERFISH_spinal_cord_resolved_0718.h5ad'
rna = sc.read_h5ad(rna_path)
rows = []
base = '/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/tacco/0503_F4_C_output'
unique_section = rna.obs['Section ID'].unique()
selected_0503 = [s for s in unique_section if s.startswith('0503')]
selected_0503_clean = [s for s in selected_0503 if s != "0503_nan_nan"]
selected_0503_1 = [s for s in selected_0503_clean if s != "0503_F4_C"]
# Output base path
outdir = "/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/tacco/0503_F4_C_output"
os.makedirs(outdir, exist_ok=True)
puck = rna[rna.obs['Section ID'] == '0503_F4_C'].copy()
# k in range(2, len(selected_0503_1)+1):
for k in range(1,18):
    subset_ids = selected_0503_1[:k]

    print(f"\n>>> Running Tangram for first {k} sections: {subset_ids}")

    # Subset total data for these slices
    adata_subset = rna[rna.obs['Section ID'].isin(subset_ids)].copy()
    puck = make_float32(puck)
    adata_subset = make_float32(adata_subset)

# (Optional) If your counts actually live in a layer, move it into X *as float32*
    if 'counts' in puck.layers:
        L = puck.layers['counts']
        puck.X = (L.asfptype().astype(np.float32) if sp.issparse(L)
              else np.asarray(L, dtype=np.float32))

    if 'counts' in adata_subset.layers:
        L = adata_subset.layers['counts']
        adata_subset.X = (L.asfptype().astype(np.float32) if sp.issparse(L)
                      else np.asarray(L, dtype=np.float32))

    print("puck.X dtype:", puck.X.dtype)
    print("adata_subset.X dtype:", adata_subset.X.dtype)

# Force TACCO to read from X (not a layer)
    import tacco as tc
    tc.tl.annotate(
        puck,
        adata_subset,
        'MERFISH cell type annotation',
        result_key='ClusterName',
        counts_location='X',   # be explicit
    )
    import pandas as pd
    tmp = puck.obsm['ClusterName']
    max_col = tmp.idxmax(axis=1).rename('max_col')
    max_val = tmp.max(axis=1).rename('max_val')
    out = pd.concat([max_col, max_val], axis=1) 
    out.index = out.index.astype(puck.obs.index.dtype)
    merged = puck.obs.join(out, how='left') 
    ft = merged["max_col"]
    mc = merged["MERFISH cell type annotation"]
    a = merged['MERFISH cell type annotation'].astype('category')
    b = merged['max_col'].astype('category')

    # unify (unordered) categories by taking the union
    cats = a.cat.categories.union(b.cat.categories)
    a = a.cat.set_categories(cats)
    b = b.cat.set_categories(cats)
    from sklearn.metrics import adjusted_rand_score
    ari = adjusted_rand_score(merged["max_col"], merged["MERFISH cell type annotation"])
    num_equal = (a == b).sum()
    total = len(merged)
    ratio = num_equal / total

    rows.append({
        "series": k,
        "ACC": ratio,
        "ARI": ari
    })
    outfile = os.path.join(outdir, f"TACCO_first{k}_slices.csv")
    merged.to_csv(outfile)
summary = pd.DataFrame(rows).set_index("series").sort_index()
out_csv = osp.join(base, "TACCO_summary.csv")
summary.to_csv(out_csv)
print(summary)


>>> Running Tangram for first 1 sections: ['0503_F5_T']
puck.X dtype: float32
adata_subset.X dtype: float32
Starting preprocessing
Annotation profiles were not found in `reference.varm["MERFISH cell type annotation"]`. Constructing reference profiles with `tacco.preprocessing.construct_reference_profiles` and default arguments...
Finished preprocessing in 0.03 seconds.
Starting annotation of data with shape (6602, 500) and a reference of shape (1927, 500) using the following wrapped method:
+- platform normalization: platform_iterations=0, gene_keys=MERFISH cell type annotation, normalize_to=adata
   +- multi center: multi_center=None multi_center_amplitudes=True
      +- bisection boost: bisections=4, bisection_divisor=3
         +- core: method=OT annotation_prior=None
mean,std( rescaling(gene) )  3.2744113465127254 1.5490375426494167
bisection run on 1
bisection run on 0.6666666666666667
bisection run on 0.4444444444444444
bisection run on 0.2962962962962963
bisection run on 0.1975